<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EA%B0%9C%EB%A1%A0_6%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

문제 1 (주관식)
- 회귀(Regression) 문제에서 MSE(평균 제곱 오차) 손실 함수가 MAE(평균 절대 오차) 손실 함수와 비교하여 갖는 가장 큰 특징(단점)을 이상치(Outlier)와 연관지어 1~2줄로 간략히 서술하시오.

MSE는 MAE에 비해 이상치에 굉장히 민감하다

문제 2(주관식)

이진 분류(Binary Classification) 문제에서 nn.BCEWithLogitsLoss()를 사용하는 주된 이유를 수치적 안정성(Numerical Stability) 관점에서 1~2줄로 간략히 서술하시오.

Sigmoid와 BCE를 하나로 합쳐 안정성을 보장하여 수치 불안정을 예방한다

문제 3 (실습 문제 - 코드 작성)

- 파이토치의 nn.Module을 상속받아 MAE (Mean Absolute Error, L1 Loss) 손실 함수 역할을 하는 커스텀 손실 클래스 CustomMAELoss를 정의하시오.

- 요구사항
  -  forward 메소드는 예측값 텐서 yp와 실제값 텐서 yt를 인자로 받습니다.
  - torch.abs()를 사용하여 오차의 절댓값을 계산합니다.
  - .mean()을 사용하여 절댓값 오차의 평균을 계산하여 반환합니다.

- MAE 공식: $\text{loss} = \frac{1}{n} \sum_{i=1}^{n} |yp_i - yt_i|$

In [2]:
import torch
import torch.nn as nn

class CustomMAELoss(nn.Module):
    def __init__(self):
        super(CustomMAELoss, self).__init__()

    def forward(self, yp, yt):
        # TODO: 여기에 MAE 손실을 계산하는 코드를 작성하세요
        loss = (torch.abs(yp - yt)).mean()
        return loss

# --- 테스트 코드 (수정 불필요) ---
criterion_mae = CustomMAELoss()
yp_test = torch.tensor([1.0, 2.0, 3.0])
yt_test = torch.tensor([1.5, 2.5, 5.0])
# 예상 MAE = (|1.0-1.5| + |2.0-2.5| + |3.0-5.0|) / 3 = (0.5 + 0.5 + 2.0) / 3 = 3.0 / 3 = 1.0
test_loss = criterion_mae(yp_test, yt_test)
print(f"커스텀 MAE 손실: {test_loss:.4f}")

커스텀 MAE 손실: 1.0000


문제 4 (실습 문제 - 코드 작성)

- Huber Loss를 파이토치 함수를 이용해 huber_loss(yp, yt, delta=1.0) 함수로 구현하시오.

- Huber Loss 공식 (델타=1.0 기준):
  - $|yp - yt| \le 1$ 이면 (오차가 작으면), $0.5 \times (yp - yt)^2$ (MSE처럼)
  - $|yp - yt| > 1$ 이면 (오차가 크면), $1.0 \times (|yp - yt| - 0.5 \times 1.0)$ (MAE처럼)
  
- 요구사항
  - torch.where(condition, a, b) 함수를 사용하면 편리합니다.
  - torch.abs()를 사용하여 절댓값 오차 err를 먼저 계산합니다.

In [3]:
import torch
import torch.nn as nn

def huber_loss(yp, yt, delta=1.0):
    # TODO: 여기에 Huber Loss 계산 코드를 작성하세요

    # 1. 오차의 절댓값 계산
    err = torch.abs(yp - yt)

    # 2. 오차가 delta보다 작은지(True/False) 조건 계산
    condition = err <= delta

    # 3. torch.where를 사용하여 조건에 따라 MSE 부분과 MAE 부분 계산
    # MSE 부분: 0.5 * err^2
    # MAE 부분: delta * (err - 0.5 * delta)
    loss_per_element = torch.where(
        condition,
        0.5 * err**2,
        delta * (err - 0.5 * delta))

    # 4. 전체 배치의 평균 손실 반환
    return loss_per_element.mean()

# --- 테스트 코드 (수정 불필요) ---
# (참고: PyTorch 내장 함수 nn.HuberLoss(delta=1.0)와 결과가 동일해야 함)
criterion_huber = huber_loss
yp_test = torch.tensor([1.0, 2.0, 3.0, 5.0])
yt_test = torch.tensor([1.5, 2.5, 1.0, 8.0])
# 오차: [-0.5, -0.5, 2.0, -3.0]
# 절댓값 오차: [0.5, 0.5, 2.0, 3.0]
# 손실 (delta=1.0):
#   0.5 <= 1.0 -> 0.5 * 0.5^2 = 0.125
#   0.5 <= 1.0 -> 0.5 * 0.5^2 = 0.125
#   2.0 > 1.0  -> 1.0 * (2.0 - 0.5 * 1.0) = 1.5
#   3.0 > 1.0  -> 1.0 * (3.0 - 0.5 * 1.0) = 2.5
# 평균 손실 = (0.125 + 0.125 + 1.5 + 2.5) / 4 = 4.25 / 4 = 1.0625
test_loss = criterion_huber(yp_test, yt_test, delta=1.0)
print(f"커스텀 Huber 손실: {test_loss:.4f}")

커스텀 Huber 손실: 1.0625


문제 5 (코드 작성)

이진 분류 문제에서 nn.BCEWithLogitsLoss()를 손실 함수로 사용하여 학습하는 코드의 빈칸을 완성하시오.

요구사항:
1. nn.Linear를 사용하여 입력 5, 출력 1을 갖는 model을 정의합니다. (이진 분류는 출력이 1개)
2. nn.BCEWithLogitsLoss()를 **criterion**으로 정의합니다. (이름에 주의!)
3. 모델의 출력(logits)과 실제 레이블(Y)을 criterion에 전달하여 loss를 계산합니다.

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 데이터 (수정 불필요) ---
# 입력 특성 5개, 이진 레이블 (0 또는 1)
X = torch.randn(10, 5)
Y = torch.tensor([1., 0., 1., 0., 1., 1., 0., 0., 1., 1.]).view(-1, 1) # (10, 1) 크기

# TODO: 1. 모델 정의 (input_dim=5, output_dim=1)
model = nn.Linear(5,1)

# TODO: 2. 손실 함수 정의 (BCEWithLogitsLoss)
criterion = nn.BCEWithLogitsLoss()

# 옵티마이저 (수정 불필요)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# --- 1회 학습 스텝 (수정 불필요) ---
# 1. 예측 (Sigmoid를 거치지 않은 원시 출력, Logits)
logits = model(X)

# TODO: 3. 손실 계산
loss = criterion(logits, Y)

# 4. 경사 계산 및 업데이트 (수정 불필요)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# --- 결과 확인 (수정 불필요) ---
print(f"모델 출력 (Logits) 예시:\n{logits.data[:3]}")
print(f"계산된 손실 (BCEWithLogitsLoss): {loss.item():.4f}")

모델 출력 (Logits) 예시:
tensor([[-0.0050],
        [ 0.6107],
        [ 1.0358]])
계산된 손실 (BCEWithLogitsLoss): 0.7124


문제 6 (실습 문제 - 코드 작성)
-  라벨 스무딩(Label Smoothing)이 적용된 nn.CrossEntropyLoss를 손실 함수로 사용하여 다중 분류 모델을 학습하는 코드의 빈칸을 완성하시오.

요구사항:
1. nn.Linear를 사용하여 입력 10, 출력 3을 갖는 **model**을 정의합니다. (3개 클래스 분류)
2. label_smoothing=0.1 옵션을 적용하여 nn.CrossEntropyLoss를 **criterion**으로 정의합니다.
3. 모델의 출력(logits)과 실제 레이블(Y)을 criterion에 전달하여 **loss**를 계산합니다. (참고: CE Loss는 Y가 클래스 인덱스 형태(1차원)이길 기대함)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- 데이터 (수정 불필요) ---
# 입력 특성 10개, 다중 클래스 레이블 (0, 1, 2)
X = torch.randn(8, 10)
Y = torch.tensor([0, 1, 2, 1, 0, 2, 1, 0]) # 1차원 인덱스

# TODO: 1. 모델 정의 (input_dim=10, output_dim=3)
model = nn.Linear(10,3)

# TODO: 2. 손실 함수 정의 (CrossEntropyLoss, label_smoothing=0.1)
smoothing_alpha = 0.1
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 옵티마이저 (수정 불필요)
optimizer = optim.SGD(model.parameters(), lr=0.1)

# --- 1회 학습 스텝 (수정 불필요) ---
# 1. 예측 (Logits)
logits = model(X)

# TODO: 3. 손실 계산
loss = criterion(logits,Y)

# 4. 경사 계산 및 업데이트 (수정 불필요)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# --- 결과 확인 (수정 불필요) ---
print(f"모델 출력 (Logits) 예시:\n{logits.data[:2]}")
print(f"계산된 손실 (Label Smoothing CE): {loss.item():.4f}")

모델 출력 (Logits) 예시:
tensor([[-0.0876,  0.0379,  0.5909],
        [ 0.9588,  0.0062,  0.5882]])
계산된 손실 (Label Smoothing CE): 1.4054
